# *Threading*

## 1. Introducción y Conceptos Básicos

* **Limitación secuencial:** Hasta el uso de *threads*, los programas escritos ejecutan una sola secuencia de instrucciones a la vez, esperando a que una tarea termine para pasar a la siguiente.
* **Procesos:** Un proceso es un programa en ejecución que ocupa un espacio en la memoria RAM y ejecuta un flujo de instrucciones. Todo proceso tiene al menos un *thread* principal.
* **Núcleos (Cores):** La CPU de un computador lee y ejecuta las instrucciones mediante sus núcleos. Sistemas modernos tienen múltiples núcleos, lo que permite que distintos procesos se ejecuten de manera realmente simultánea.
* **Paralelismo vs. Concurrencia:**
    * **Paralelismo:** Ejecución simultánea de varias tareas en diferentes núcleos o procesadores (hardware separado).
    * **Concurrencia:** Gestión de varias tareas de forma intercalada en un mismo núcleo. Los cambios de contexto son tan rápidos que aparentan simultaneidad.
* **El GIL en Python:** Python utiliza el *Global Interpreter Lock* (GIL), el cual evita ejecutar *threads* en paralelo real, asegurando que solamente un *thread* pueda ejecutar código Python en un instante dado.

## 2. *Threading* en Python

Un *thread* (o hilo) es una unidad de ejecución de código dentro de un proceso que lleva la cuenta de en qué parte del código estamos. Sus usos comunes incluyen interfaces gráficas, aplicaciones multiusuario y el modelo productor-consumidor.

### 2.1 Creación de *Threads*
Se manejan mediante el módulo `threading`. Se utiliza la clase `Thread`, pasándole la función a ejecutar en el parámetro `target`.
* Los *threads* **NO** se ejecutan automáticamente al instanciarse; se debe llamar al método `start()`.
* Son de un solo uso: si se intenta iniciar un *thread* que ya inició, levantará un `RuntimeError`.
* Se pueden pasar argumentos a la función objetivo usando `args` (tupla) o `kwargs` (diccionario).

In [ ]:
import threading
import time

def saludar_persona(nombre):
    print(f"[{threading.current_thread().name}] Hola {nombre}! Empezando tarea...")
    time.sleep(1)
    print(f"[{threading.current_thread().name}] Adiós {nombre}! Tarea terminada.")

hilo1 = threading.Thread(name="Hilo-1", target=saludar_persona, args=("Alice",))
hilo2 = threading.Thread(name="Hilo-2", target=saludar_persona, kwargs={"nombre": "Bob"})

hilo1.start()
hilo2.start()

print("[MainThread] Los hilos están corriendo independientemente de mí.")

### 2.2 *Threads* basados en Programación Orientada a Objetos (OOP)
Podemos crear clases que hereden de `threading.Thread` para personalizar el comportamiento. Se debe sobreescribir el método `run()` (que define las instrucciones a ejecutar) y llamar al `__init__` original con `super().__init__()`.

In [ ]:
class Trabajador(threading.Thread):
    def __init__(self, nombre, tiempo):
        super().__init__(name=nombre) # Llamada obligatoria al init original
        self.tiempo = tiempo
        
    def run(self):
        print(f"{self.name} trabajando por {self.tiempo} seg...")
        time.sleep(self.tiempo)
        print(f"{self.name} terminó su trabajo.")

trabajador_oop = Trabajador("HiloOOP", 2)
trabajador_oop.start()

### 2.3 Sincronización y Estados: `join()` e `is_alive()`
* **`join(timeout=None)`:** El *thread* que llama a este método se bloquea y espera hasta que el *thread* referenciado termine. Si se usa el parámetro `timeout`, esperará como máximo esa cantidad de segundos.
* **`is_alive()`:** Retorna un booleano indicando si el *thread* sigue en funcionamiento en ese instante. Es útil para evaluar el estado luego de un `join()` con límite de tiempo.

In [ ]:
lento = Trabajador("Lento", 5)
rapido = Trabajador("Rápido", 1)

lento.start()
rapido.start()

rapido.join() # Espera a que termine por completo
print("[MainThread] Hilo rápido terminó. Esperando al lento máximo 2 segs...")
lento.join(2)

if lento.is_alive():
    print("[MainThread] El hilo lento aún sigue trabajando, no lo voy a esperar más.")
else:
    print("[MainThread] El hilo lento alcanzó a terminar.")

### 2.4 *Daemon Threads*
Los *daemon threads* son aquellos que corren en segundo plano y no impiden que el programa principal termine.
* Si el programa principal acaba, los *daemon threads* son interrumpidos abruptamente.
* Se declaran con el parámetro `daemon=True` en la instanciación o modificando el atributo `daemon` antes de ejecutar `start()`.
* **Importante:** Una vez iniciado el *thread*, intentar cambiar su estado a/desde *daemon* genera un `RuntimeError`.

In [ ]:
def tarea_infinita():
    while True:
        print("[Daemon] Sigo vivo...")
        time.sleep(1)

# Forma de instanciar un Daemon
hilo_fondo = threading.Thread(target=tarea_infinita, daemon=True)
hilo_fondo.start()
time.sleep(3)
print("[MainThread] Termino mi ejecución. El Daemon morirá abruptamente al ejecutar esto en un script.")

### 2.5 *Timers*
La clase `Timer` es una subclase de `Thread` que permite ejecutar una acción después de que ha pasado un tiempo determinado.
* Requiere especificar el tiempo (en segundos) y la función a ejecutar.
* Posee un método `.cancel()` que permite cancelar la ejecución si se invoca antes de que el *timer* comience su tarea.

In [ ]:
def alarma(mensaje):
    print(f"\n¡RING RING! {mensaje}")

timer_1 = threading.Timer(4.0, alarma, args=("Ya pasaron 4 segundos",))
timer_2 = threading.Timer(10.0, alarma, args=("No verás esto",))

timer_1.start()
timer_2.start()

# Cancelamos el timer_2 antes de que transcurran sus 10 segundos
timer_2.cancel()
print("Timer 2 cancelado con éxito.")

---
## 3. Ejercicios Propuestos

### Carrera de Contadores
Crea dos *threads*. El primero debe imprimir los números del 1 al 5, esperando 0.5 segundos entre cada número. El segundo debe imprimir las letras de la 'A' a la 'E', esperando también 0.5 segundos entre cada letra. El hilo principal debe iniciar ambos hilos y luego imprimir "¡Carrera iniciada!".

### Servidor Descargador
Diseña una clase `Descargador` que herede de `threading.Thread`. Su método `run` debe simular la descarga de un archivo haciendo un `time.sleep(tiempo_descarga)` (donde `tiempo_descarga` se recibe al instanciar el objeto). 
1. Crea 3 descargadores con tiempos de 2, 4 y 6 segundos respectivamente.
2. Inicia todos los hilos.
3. Usa un `.join()` para esperar a todos simultáneamente (piensa bien cómo estructurar los joins para no bloquearlos secuencialmente de manera ineficiente).
4. Al finalizar todos, el hilo principal debe imprimir "Todas las descargas han finalizado".


# Concurrencia y Consecuencias

---

## 1. El Problema de la Sincronización

Cuando utilizamos *threads* para realizar tareas de forma pseudo-paralela, podemos encontrarnos con un grave problema si múltiples *threads* intentan modificar un mismo recurso (como una variable compartida) al mismo tiempo. 

Las operaciones no siempre son **atómicas**. Por ejemplo, sumar 1 a un contador (`contador.valor += 1`) se descompone en varios pasos a nivel de máquina: leer el valor, sumar 1, y guardar el nuevo valor. Como el Sistema Operativo puede pausar un *thread* en cualquier momento para darle paso a otro, los pasos de distintos *threads* se pueden entremezclar, causando que se sobrescriban valores y se pierda información.

A la porción de código donde accedemos y modificamos este recurso compartido, y en la cual **solo debería haber un *thread* ejecutándose a la vez**, se le llama **sección crítica**.

## 2. Mecanismos de Sincronización: *Locks*

Para proteger nuestra **sección crítica**, usamos una primitiva llamada *Lock* (provista por `threading.Lock`). Funciona como una llave única:

* Un *thread* debe adquirir el lock usando `acquire()` antes de entrar a la sección crítica. Si otro *thread* ya lo tiene, el actual debe esperar.
* Una vez terminada la operación, el *thread* debe liberar el lock con `release()`, permitiendo que otro entre.

**Regla de oro:** El objeto *Lock* debe ser **compartido** por todos los *threads* involucrados. Si cada *thread* crea su propio lock, no habrá protección alguna.

En Python, la forma más recomendada y segura de usar locks es mediante los **Context Managers** (la sentencia `with`). Esto maneja automáticamente el `acquire()` y el `release()`.

In [ ]:
import threading

contador = 0
# Creamos UN SOLO lock para compartir entre los threads
lock_global = threading.Lock()

def sumar_seguro():
    global contador
    for _ in range(100000):
        # El 'with' se encarga de acquire() y release()
        with lock_global:
            # --- Sección crítica ---
            contador += 1
            # --- Fin sección crítica ---

t1 = threading.Thread(target=sumar_seguro)
t2 = threading.Thread(target=sumar_seguro)

t1.start()
t2.start()
t1.join()
t2.join()

print("Contador seguro final:", contador) # Siempre dará 200000

## 3. Señales entre *threads*: `Event`

A veces no solo queremos evitar que dos *threads* choquen, sino que queremos que se **coordinen**. Por ejemplo, un *thread* debe esperar a que otro termine de cargar un archivo antes de continuar. 

Para esto usamos los `Event`. Un evento tiene una bandera interna (True/False) e interactuamos con él a través de métodos:

* `wait()`: Pausa el *thread* actual hasta que el evento sea verdadero.
* `set()`: Vuelve la bandera verdadera (¡activa la señal!). Esto despierta a todos los *threads* que estaban haciendo `wait()`.
* `clear()`: Vuelve la bandera a falso (resetea la señal).
* `is_set()`: Retorna `True` o `False` indicando el estado de la señal, sin pausar el *thread*.

In [ ]:
import threading
import time

datos_listos = threading.Event()

def procesador():
    print("Procesador: Esperando a que el lector termine...")
    datos_listos.wait() # Se pausa aquí hasta recibir la señal
    print("Procesador: ¡Señal recibida! Procesando datos...")

def lector():
    print("Lector: Leyendo datos pesados...")
    time.sleep(2) # Simulamos tiempo de lectura
    print("Lector: Terminé. Avisando al procesador.")
    datos_listos.set() # Activamos la señal

t_procesador = threading.Thread(target=procesador)
t_lector = threading.Thread(target=lector)

t_procesador.start()
t_lector.start()
t_procesador.join()
t_lector.join()

## 4. Deadlocks (Interbloqueos)

Un *deadlock* es una situación fatal donde dos o más *threads* se quedan esperando mutuamente para siempre y el programa nunca avanza. 

Puede ocurrir por:
1.  **Múltiples Locks:** Si el Thread A adquiere el Lock 1 y espera el Lock 2, pero el Thread B adquirió el Lock 2 y espera el Lock 1.
2.  **Múltiples Eventos:** Si el Thread A espera el evento del Thread B antes de mandar su propia señal, pero el Thread B hace exactamente lo mismo respecto al Thread A.

In [ ]:
# EJEMPLO DE DEADLOCK CON LOCKS (NO EJECUTAR SIN PRECAUCIÓN)
import threading
import time

lock_1 = threading.Lock()
lock_2 = threading.Lock()

def func_a():
    with lock_1:
        time.sleep(0.5)
        # Intenta pedir el lock_2, pero func_b lo tiene.
        # with lock_2:
        #    print("A terminó")

def func_b():
    with lock_2:
        time.sleep(0.5)
        # Intenta pedir el lock_1, pero func_a lo tiene.
        # with lock_1:
        #    print("B terminó")

print("El código comentado muestra cómo un deadlock congela el programa.")

---
# Ejercicios Propuestos

### Ejercicio 1 (Básico): Turnos estrictos
Crea dos *threads*, uno llamado `Ping` y otro `Pong`. Deben imprimir sus nombres de forma alternada estrictamente 3 veces cada uno (es decir: Ping, Pong, Ping, Pong, Ping, Pong). Usa `Event` para coordinar de quién es el turno.

In [ ]:
import threading

# Escribe tu código aquí para el Ejercicio 1



### Ejercicio 2 (Intermedio): El Cajero Compartido
Imagina una cuenta bancaria con 1000. Crea una función `extraer_dinero(monto)` y una función `depositar_dinero(monto)`. Crea 4 *threads*: dos que intenten extraer 300 repetidamente (3 veces) y dos que intenten depositar $100 repetidamente (3 veces). Usa un `Lock` para asegurar que el balance final sea matemáticamente consistente al final de la ejecución.

In [ ]:
import threading
import time

cuenta_bancaria = 1000

# Escribe tu código aquí para el Ejercicio 2



### Ejercicio 3 (Avanzado): Escapando del Deadlock
El siguiente código causa un *Deadlock* utilizando eventos, similar al caso del trabajador y el jefe visto en la materia original. Analiza la lógica e identifica por qué se están esperando mutuamente. Luego, **modifica el código** para arreglarlo de manera que el sistema no se quede pegado y ambos *threads* puedan completar su trabajo e imprimir su mensaje final.

In [ ]:
import threading
import time

paso_1_listo = threading.Event()
paso_2_listo = threading.Event()

def componente_x():
    print("Componente X: Iniciando... y esperando a Y.")
    paso_2_listo.wait() 
    
    print("Componente X: Trabajando...")
    time.sleep(1)
    paso_1_listo.set() 
    print("Componente X: ¡Terminado!")

def componente_y():
    print("Componente Y: Iniciando... y esperando a X.")
    paso_1_listo.wait() 
    
    print("Componente Y: Trabajando...")
    time.sleep(1)
    paso_2_listo.set()
    print("Componente Y: ¡Terminado!")

t_x = threading.Thread(target=componente_x)
t_y = threading.Thread(target=componente_y)

# Ejecuta esto bajo tu propio riesgo (se quedará pegado hasta que lo arregles)
# t_x.start()
# t_y.start()
# t_x.join()
# t_y.join()


# Threads y Concurrencia

---

## 1. Lock como atributo de clase

Al trabajar con subclases de `threading.Thread`, es una buena práctica definir los objetos `Lock` como **atributos de clase**. 

### Conceptos Clave:
* **Atributo de Instancia:** Se define con `self` y es único para cada objeto.
* **Atributo de Clase:** Se define fuera de los métodos. Es compartido por todas las instancias de esa clase.
* **Orden de búsqueda:** Al usar `self.atributo`, Python busca primero en la instancia, luego en la clase y finalmente lanza `AttributeError`.

### Ejemplo: Escritura compartida en archivos
Si múltiples hilos intentan escribir en un mismo archivo simultáneamente, los datos podrían corromperse. Usar un `Lock` de clase asegura que solo un hilo acceda al archivo a la vez.

In [ ]:
import threading
import os

class Escritor(threading.Thread):
    # Lock compartido por todas las instancias de Escritor
    lock_archivo = threading.Lock()

    def __init__(self, mensaje):
        super().__init__()
        self.mensaje = mensaje

    def run(self):
        with Escritor.lock_archivo:
            with open("salida.txt", "a") as f:
                f.write(f"{self.mensaje}\n")

## 2. Patrón Productor-Consumidor

Este patrón ocurre cuando hilos (**productores**) generan datos y los colocan en un *buffer*, mientras que otros (**consumidores**) los extraen para procesarlos.

### Sincronización con `deque` vs `Queue`:
1.  **`collections.deque`**: Es *thread-safe* para agregar/quitar elementos, pero no garantiza que la revisión de contenido y la extracción sean atómicas sin un `Lock` externo.
2.  **`queue.Queue`**: Está diseñada específicamente para hilos. El método `.get()` es **bloqueante**: si la cola está vacía, el hilo espera automáticamente hasta que haya un elemento disponible, ahorrando ciclos de CPU.

In [ ]:
from queue import Queue
import threading
import time

buffer = Queue()

def productor():
    for i in range(5):
        time.sleep(1)
        buffer.put(f"Dato {i}")
        print(f"Producido: {i}")

def consumidor():
    while True:
        item = buffer.get() # Espera automáticamente si está vacía
        print(f"Consumido: {item}")
        buffer.task_done()

threading.Thread(target=productor).start()
threading.Thread(target=consumidor, daemon=True).start()

## 3. Simulaciones y Eventos

Los hilos permiten modelar entidades con comportamientos simultáneos. Para la comunicación entre ellos, se utilizan objetos `threading.Event`:
* `.set()`: Activa la señal.
* `.wait()`: Pausa el hilo hasta que la señal sea activada.
* `.clear()`: Desactiva la señal.

Esto es ideal para modelos de "pregunta-respuesta" (como el ejemplo de Estudiante-Profesor).

---
# Ejercicios propuestos

### El Contador Seguro
Crea una clase `Contador` que herede de `Thread`. Define un atributo de clase llamado `valor = 0` y un `Lock`. Crea 10 hilos que aumenten el valor en 1 cada uno. Usa el `Lock` para evitar condiciones de carrera al aumentar el contador.

### Sistema de Pedidos (Productor-Consumidor)
Simula una pizzería. 
1. Un hilo `Cocinero` que pone "Pizzas" en una `Queue` cada cierto tiempo aleatorio.
2. Tres hilos `Repartidor` que sacan las pizzas de la cola y simulan la entrega con un `time.sleep()`.
3. Asegúrate de que los repartidores esperen correctamente si no hay pizzas listas.

### Simulación de Cruce de Peatones
Diseña una simulación con `Event`.
1. Un hilo `Semaforo` que alterna entre "Verde" y "Rojo" cada 5 segundos.
2. Múltiples hilos `Peaton` que llegan al cruce. 
3. Si el semáforo está en rojo, los peatones deben esperar (`.wait()`). 
4. Cuando el semáforo cambia a verde, todos los peatones que estaban esperando deben cruzar simultáneamente y continuar su camino.